# Parda — text model v2, part B (GPU): EasyOCR + train GLiNER v2 + re-run both benchmarks

v1 learned only from clean text and Tesseract's mistakes on scans. v2 also learns from **phone photos**
(Tesseract) and from **EasyOCR's** readings (line-level, labels carried over by text alignment), because the
pipeline reads every page with both. The 300 benchmark pages and the 4 real-world-test layouts stay unseen.

**Settings:** Accelerator **GPU T4** (x1 or x2) · Internet **On**
**Inputs (Add Input → Your Work):** your **Parda** (Phase 1) notebook · your **Phase 3a** notebook (OCR training
file) · the **part A** notebook (photo pages) · your **Phase 5** notebook · your **real-world test** notebook (06)
· **Datasets tab →** `celeba-dataset`. **Secrets:** `GITHUB_TOKEN`, `HF_TOKEN`

Run cells 1–4 by hand (~5 min), then **Save Version → Save & Run All** (~3 h). You can close the browser.


In [ ]:
# 1. Code + secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)


In [ ]:
# 2. Fonts, Tesseract + EasyOCR, GLiNER, YOLO
!bash scripts/setup_kaggle.sh
!bash scripts/setup_ocr.sh
!pip install -q gliner ultralytics
!python -c "import gliner, torch; print('gliner', gliner.__version__, '| GPU:', torch.cuda.get_device_name(0))"


In [ ]:
# 3. Find every input (fast: never lists the 200k face photos)
import json, math, shutil
from huggingface_hub import HfApi
from parda.fsutil import find_path
from parda.ocr.run_ocr import find_data, _n_images
from parda.pii.build_data import find_ocr_train
from parda.vision.faces import FacePool, find_faces_dir
IN, W = "/kaggle/input", "/kaggle/working"
first = lambda rel: (find_path(IN, rel) or [None])[0]
DATA = find_data()
OCR_SCAN = find_ocr_train()
PHOTO = os.path.dirname(first("photo/ocr_tesseract.jsonl") or "")
C_SEEN, C_OOD = first("e2e_cache/ocr_tesseract.jsonl"), first("ood_cache/ocr_tesseract.jsonl")
V1_SEEN, V1_OOD = first("e2e_eval/report_e2e.json"), first("ood_eval/report_e2e.json")
FACES = find_faces_dir(IN, exclude=[DATA, PHOTO])
HF_USER = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
BASE, YOLO_REPO = "urchade/gliner_multi_pii-v1", f"{HF_USER}/parda-yolo-v1"
for k, v in [("DATA (Phase 1)", DATA), ("OCR_SCAN (Phase 3a)", OCR_SCAN), ("PHOTO (part A)", PHOTO), ("FACES", FACES),
             ("OCR cache seen (Phase 5)", C_SEEN), ("OCR cache unseen (06)", C_OOD),
             ("v1 report seen", V1_SEEN), ("v1 report unseen", V1_OOD)]:
    print(f"{k:26s} {v}")
assert DATA and _n_images(DATA) == 30000, "attach your Parda (Phase 1) notebook"
assert OCR_SCAN, "attach your Phase 3a notebook (ocr_train/ocr_tesseract.jsonl)"
assert PHOTO, "attach the part A notebook (photo/ocr_tesseract.jsonl)"
assert FACES, "Add Input -> Datasets tab -> 'celeba-dataset'"
if not (C_SEEN and C_OOD):
    print("NOTE: an OCR cache is missing -> the benchmarks will redo OCR (~20 min extra). Attach the Phase 5 / 06 notebooks.")


In [ ]:
# 4. Tests (~1 min): every line ok
!python tests/test_v2.py


In [ ]:
# 5. EasyOCR on 1,200 photo pages + 1,200 scan pages (~50 min, GPU; resumable)
!python -m parda.pii.ocr_pages --data {PHOTO} --engine easyocr --per_lang 400 --seed 31 --bench_of {DATA} --out_dir {W}/easy_photo 2>&1 | tail -2
!python -m parda.pii.ocr_pages --data {DATA} --engine easyocr --per_lang 400 --seed 32 --out_dir {W}/easy_scan 2>&1 | tail -2


In [ ]:
# 6. Build the v2 training data: clean text + Tesseract (scans, photos) + EasyOCR (scans, photos) (~5 min)
!python -m parda.pii.build_data --data {DATA} --noisy tess_scan={DATA}:{OCR_SCAN} --noisy tess_photo={PHOTO}:{PHOTO}/ocr_tesseract.jsonl --noisy easy_scan={DATA}:{W}/easy_scan/ocr_easyocr.jsonl:lines --noisy easy_photo={PHOTO}:{W}/easy_photo/ocr_easyocr.jsonl:lines --clean_docs 10000 --out {W}/gliner_data_v2 --workers 4 > /dev/null
st = json.load(open(f"{W}/gliner_data_v2/stats.json"))
print(json.dumps({k: st[k] for k in ("train_examples", "val_examples", "train_by_source", "transfer_by_source")}, indent=2))


In [ ]:
# 7. SMOKE TEST (~2 min): 10 steps, the loss must be a number
!python -m parda.pii.train_gliner --data_dir {W}/gliner_data_v2 --out /tmp/gliner_smoke_v2 --base {BASE} --smoke 2>&1 | tail -12
s = json.load(open("/tmp/gliner_smoke_v2/train_summary.json"))
assert s["last_loss"] is not None and math.isfinite(s["last_loss"]), s
print("smoke OK, last loss", round(s["last_loss"], 2))


In [ ]:
# 8. TRAIN GLiNER v2: 1 epoch over all the data (~75 min), uploaded to a private HF repo
n_train = st["train_examples"]
STEPS = math.ceil(n_train / 8)
print("train examples", n_train, "-> steps", STEPS)
!python -m parda.pii.train_gliner --data_dir {W}/gliner_data_v2 --out {W}/gliner_v2 --base {BASE} --epochs 1 --bs 8 --max_steps {STEPS} --hf_repo {HF_USER}/parda-gliner-v2
V2 = f"{W}/gliner_v2/final"
print(json.load(open(f"{W}/gliner_v2/train_summary.json")))


In [ ]:
# 9. Benchmark on the SEEN templates (the Phase 5 pages; OCR reused from the Phase 5 cache) (~10 min)
!python -m parda.vision.yolo_data --data {DATA} --out {W}/yolo_bench --faces_dir {FACES} --only bench --workers 4 > /dev/null
os.makedirs(f"{W}/cache_seen", exist_ok=True)
if C_SEEN:
    for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
        shutil.copy(os.path.join(os.path.dirname(C_SEEN), f), f"{W}/cache_seen/{f}")
!python -m parda.pipeline.evaluate_e2e --data {DATA} --yolo_data {W}/yolo_bench --ocr tesseract,easyocr --gliner_ft {V2} --gliner_base {BASE} --yolo {YOLO_REPO} --cache {W}/cache_seen --out {W}/v2_seen --title "Text model v2 — seen templates" 2>&1 | grep -v Warning | tail -12


In [ ]:
# 10. Benchmark on the UNSEEN layouts + phone photos (the same 300 pages as notebook 06) (~10 min)
!python -m parda.synth.ood --n 300 --out {W}/ood --faces_dir {FACES} --workers 4 > /dev/null
os.makedirs(f"{W}/cache_ood", exist_ok=True)
if C_OOD:
    for f in ("ocr_tesseract.jsonl", "ocr_easyocr.jsonl"):
        shutil.copy(os.path.join(os.path.dirname(C_OOD), f), f"{W}/cache_ood/{f}")
!python -m parda.pipeline.evaluate_e2e --data {W}/ood --ocr tesseract,easyocr --gliner_ft {V2} --gliner_base {BASE} --yolo {YOLO_REPO} --cache {W}/cache_ood --out {W}/v2_ood --title "Text model v2 — unseen layouts + phone photos" 2>&1 | grep -v Warning | tail -12


In [ ]:
# 11. v1 vs v2, side by side
from IPython.display import Markdown, display
pct = lambda v: "–" if v is None else f"{100 * v:.1f}%"
def compare(v1_path, v2_path, title):
    v2 = {r["config"]: r for r in json.load(open(v2_path))["configs"]}
    v1 = {r["config"]: r for r in json.load(open(v1_path))["configs"]} if v1_path else {}
    L = [f"### {title}", "", "| Configuration | All PII v1 → v2 | Text v1 → v2 | Pages with no leak v1 → v2 | Over-redaction v1 → v2 |",
         "|---|---|---|---|---|"]
    for name, r in v2.items():
        if "fine-tuned" in name or name.startswith("+"):
            o = v1.get(name, {})
            L.append(f"| {name} | {pct(o.get('fully_redacted'))} → **{pct(r['fully_redacted'])}** | {pct(o.get('text'))} → **{pct(r['text'])}** | "
                     f"{pct(o.get('pages_clean'))} → **{pct(r['pages_clean'])}** | {pct(o.get('over_redaction'))} → {pct(r['over_redaction'])} |")
    full1, full2 = v1.get("+ YOLO visual = Parda", {}).get("by_label", {}), v2["+ YOLO visual = Parda"]["by_label"]
    L += ["", "| PII type (full Parda) | v1 | v2 |", "|---|---|---|"]
    L += [f"| {k} | {pct(full1.get(k))} | **{pct(v)}** |" for k, v in full2.items()]
    return "\n".join(L) + "\n"
report = compare(V1_SEEN, f"{W}/v2_seen/report_e2e.json", "Seen templates (Phase 5 benchmark)") + "\n" + \
         compare(V1_OOD, f"{W}/v2_ood/report_e2e.json", "Unseen layouts + phone photos")
open(f"{W}/v1_vs_v2.md", "w").write(report)
display(Markdown(report))


In [ ]:
# 12. Collect results to commit to GitHub (download from the Output tab)
R = f"{W}/v2_results"
os.makedirs(R, exist_ok=True)
for src, dst in [(f"{W}/v1_vs_v2.md", "v1_vs_v2.md"), (f"{W}/gliner_v2/train_summary.json", "train_summary.json"),
                 (f"{W}/gliner_data_v2/stats.json", "data_stats.json"), (f"{W}/v2_seen/report_e2e.md", "report_seen.md"),
                 (f"{W}/v2_seen/report_e2e.json", "report_seen.json"), (f"{W}/v2_ood/report_e2e.md", "report_unseen.md"),
                 (f"{W}/v2_ood/report_e2e.json", "report_unseen.json")]:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(R, dst))
print(sorted(os.listdir(R)))
